# Proteomics pseudobulk analysis 

In this tutorial, we reanalyse minibulk DVP data from Nordmann et al, 2024^[Nordmann, T. M. et al. Spatial proteomics identifies JAKi as treatment for a lethal skin disease. Nature 1–9 (2024) doi:10.1038/s41586-024-08061-0.]
Here, the authors dissected Keratinocytes from formalin-fixed, paraffin-embedded archived skin tissue biopsies of three types of cutaneous drug reactions with increasing severity (Healthy, maculopapular rash/MPR, drug reaction with eosinophilia and systemic symptoms/DRESS, Toxic epidermal necrolysis/TEN). 

We reanalyse the data from the original publication. We will 

1. Load the data from a DIANN report
2. Add metadata
3. Perform standard feature-level filtering 
4. Log2-transform the data to increase normality and homoskedasticity
5. Impute the data with a Gaussian imputation strategy
6. Visualize the cohort properties via unsupervised principal component analysis 
7. Perform differential expression analysis between healthy and diseased tissues.

## Modules

Here, we will leverage analysis tools built on top of the scverse-ecosystem^[Virshup, I. et al. The scverse project provides a computational ecosystem for single-cell omics data analysis. Nat Biotechnol 41, 604–606 (2023).
] and the `anndata` data structure ^[Virshup, I., Rybakov, S., Theis, F. J., Angerer, P. & Wolf, F. A. anndata: Annotated data. 2021.12.16.473007 Preprint at https://doi.org/10.1101/2021.12.16.473007 (2021).]; including alphapepttools^[Brennsteiner, V., Diedrich, L., Ben-Moshe, S., Schwörer, M. & Mann, M., alphapepttools [Computer software]. https://github.com/MannLabs/alphapepttools] a software package for the analysis of MS-proteomics data, and decoupler^[Badia-i-Mompel P., Vélez Santiago J., Braunger J., Geiss C., Dimitrov D., Müller-Dott S., Taus P., Dugourd A., Holland C.H., Ramirez Flores R.O. and Saez-Rodriguez J. 2022. decoupleR: Ensemble of computational methods to infer biological activities from omics data. Bioinformatics Advances. https://doi.org/10.1093/bioadv/vbac016], as these tools are directly interoperable with the spatialdata framework we used in the [image analysis tutorial](../image-analysis/harpy.ipynb). 

Note that openDVP^[https://coscialab.github.io/openDVP/Tutorials/T2_DownstreamProteomics.html] provides similar functionalities for the analysis of DVP data in Python, which can be checked out in [this vignette](https://coscialab.github.io/openDVP/Tutorials/T2_DownstreamProteomics.html). Other packages like MSstats^[Choi M (2014). “MSstats: an R package for statistical analysis of quantitative mass spectrometry-based proteomic experiments.” Bioinformatics, 30.] or scp^[Vanderaa Christophe and Laurent Gatto. The current state of single-cell proteomics data analysis. Current Protocols 3 (1): e658.; doi: https://doi.org/10.1002/cpz1.658 (2023).] offer similar analysis functionality in R but are incompatible with the spatialdata framework.

In [1]:
import warnings
from collections.abc import Callable

import alphapepttools as apt
import anndata as ad
import decoupler as dc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import median_abs_deviation

/Users/lucas-diedrich/mamba/envs/npdvp-at/lib/python3.13/site-packages/mudata/__init__.py:30: DeprecationWarning: The decorator_name argument is deprecated and will be removed in the future.
  register_mudata_namespace = _make_register_namespace_decorator(MuData, "mdata", "register_mudata_namespace", "numpy")
/Users/lucas-diedrich/mamba/envs/npdvp-at/lib/python3.13/site-packages/mudata/__init__.py:30: DeprecationWarning: The docstring_style argument is deprecated and will be removed in the future.
  register_mudata_namespace = _make_register_namespace_decorator(MuData, "mdata", "register_mudata_namespace", "numpy")
/Users/lucas-diedrich/mamba/envs/npdvp-at/lib/python3.13/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the so

## Functions

We define a helper function to aggregate multiple replicate measurements into a patient-level representation.

In [2]:
def agg(
    adata, group_column: str, func: Callable | None = None, layer: str | None = None
) -> ad.AnnData:
    """Pseudobulk an anndata object based on a group column in adata.obs"""
    if func is None:

        def func(x):
            return np.nanmean(x, axis=0)

    groupings = adata.obs.groupby(group_column, observed=True).indices

    rows = []
    for _, indices in groupings.items():
        with warnings.catch_warnings(record=True):
            warnings.simplefilter("always", RuntimeWarning)
            rows.append(func(adata.layers[layer][indices, :]))

    return ad.AnnData(
        X=np.stack(rows),
        obs=pd.DataFrame(index=groupings.keys()),
        var=adata.var,
    )


def mad_outlier(
    adata: ad.AnnData,
    column: str,
    *,
    nmad: float = 3,
    key_added: str = "mad_outlier",
    copy: bool = False,
) -> ad.AnnData | None:
    adata = adata.copy() if copy else adata
    data = adata.obs[column]

    median = data.median(skipna=True)
    mad = median_abs_deviation(data, nan_policy="omit")

    adata.obs[key_added] = (data < median - nmad * mad) | (data > median + nmad * mad)

    return adata if copy else None

In [3]:
COLOR_MAP = {
    "Healthy": "#999FA3",
    "MPR": "#9CBFDB",
    "DRESS": "#F9BF8F",
    "TEN": "#CBA7CE",
}
ORDER = ["Healthy", "MPR", "DRESS", "TEN"]

## Load data

We load the DIANN report into an `anndata` object and add the relevant metadata:

In [4]:
adata = apt.io.read_pg_table(
    "../../data/dvp_proteomics_keratinocytes.tsv", search_engine="diann"
)
metadata = pd.read_csv(
    "../../data/metadata_keratinocytes.tsv", sep="\t", index_col="sample_id"
)

adata = apt.pp.add_metadata(adata, metadata, axis=0)
adata.obs["sample_id"] = (
    adata.obs["patient_id"] + "-" + adata.obs["replicate_nr"].astype(str)
)

ValueError: Index sample_id invalid

MS-proteomics data is typically analysed in log-space. We backup the raw intensities and log-transform the data. 

In [ ]:
# Save raw intensities
adata.layers["raw"] = adata.X.copy()

apt.pp.nanlog(adata)

### Sample-level quality control
Sample-preparation and MS-measurements are main drivers of sample-level variation. These samples are often identifiable by having far less uniquely identified features than comparable samples. We compute the feature completeness of the samples:

In [ ]:
apt.metrics.fraction_complete(adata, axis=0)

To identify outliers, you can use the median absolute deviation-based outlier thresholds as data-driven thresholding method (e.g. 3 median absolute deviations away from the cohort-median)^[Heumos, L. et al. Best practices for single-cell analysis across modalities. Nat Rev Genet 24, 550–572 (2023).]

In [ ]:
mad_outlier(adata, "fraction_complete", nmad=3, key_added="mad_outlier__completeness")
adata.obs["mad_outlier__completeness"].value_counts(normalize=True)

None of the samples were identified as outliers based on a 3x MAD away from the median.
We can further inspect feature missingness and feature intensity distributions across all samples:

In [ ]:
fig, axm = apt.pl.create_figure(
    1, 2, gridspec_kwargs={"width_ratios": [0.2, 0.8]}, figsize=(12, 3)
)
apt.pl.violinplot(
    ax=axm[0], data=adata, grouping_column="condition", value_column="fraction_complete"
)
apt.pl.label_axes(ax=axm[0], ylabel="Fraction Complete")
axm[0].set_ylim(0, 1)


apt.pl.violinplot(
    ax=axm[1],
    data=adata.T,
    direct_columns=adata.obs.index.tolist(),
)
_ = axm[1].set_xticklabels(
    adata.obs["patient_id"] + "-" + adata.obs["replicate_nr"].astype(str), rotation=90
)  # Set the new xtick labels

axm[0].set_title("Missingness per condition")
axm[1].set_title("Intensity distribution per sample")

plt.show()

Again, both metrics are consistent across the full cohort, indicating consistent sample preparation.

### Feature-level quality control

In [ ]:
adata = apt.pp.filter_data_completeness(
    adata, group_column="condition", max_missing=0.3, action="flag", keep_strategy="any"
)

#### Contaminants

In other cases, clinical samples might be contaminated with blood or classical contaminants like Keratins. We do not detect blood in the provided samples and Keratins are expected features, as we are working with skin samples. Thus, we proceed with the analysis.

In [ ]:
assert "HBB" not in adata.var["genes"]

### Aggregate

The original publication aggregated replicates per patient. We mean-aggregate the replicates per patient and log-transform the data, which we will use for downstream analysis.

In [ ]:
adata_agg = agg(adata, group_column="patient_id", layer="raw")

# Add metadata
# Subset to patient-level metadata
patient_metadata = (
    metadata[["patient_id", "condition"]]
    .drop_duplicates()
    .set_index("patient_id", drop=False)
    .rename_axis(index=None)
)
adata_agg = apt.pp.add_metadata(adata_agg, incoming_metadata=patient_metadata, axis=0)

adata_agg.layers["raw"] = adata_agg.X.copy()
apt.pp.nanlog(adata_agg)

### Feature-level filtering

In [ ]:
apt.metrics.number_detected(adata_agg)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
sns.boxplot(
    adata_agg.obs,
    x="condition",
    y="number_detected",
    palette=COLOR_MAP.values(),
    order=ORDER,
    fliersize=0,
)
sns.stripplot(
    adata_agg.obs,
    x="condition",
    y="number_detected",
    ax=plt.gca(),
    palette=COLOR_MAP.values(),
    order=ORDER,
    edgecolor="#222222",
    linewidth=1,
)
plt.gca().set(ylim=(0, 4250), yticks=np.arange(0, 4100, 1000))
plt.show()

In [ ]:
# Subset to proteins with >30% completeness in any condition
adata_agg = adata_agg[:, adata_agg.var["passed_threshold_missing_values"]].copy()

### Imputation

- Imputation is necessary for analysis steps like principal component analysis
- We recommend to work with unimputed data whenever possible. 

In [ ]:
adata_agg.layers["imputed"] = adata_agg.X.copy()
apt.pp.impute_gaussian(adata_agg, layer="imputed")

## PCA

To investigate the latent structure of the data, we compute a principal component analysis:

In [ ]:
apt.tl.pca(adata_agg, layer="imputed")

In [ ]:
fig, axm = apt.pl.create_figure(1, 1, figsize=(3, 3))
apt.pl.plot_pca(
    adata_agg,
    method="pca",
    ax=axm[0],
    color_map_column="condition",
    color_dict=COLOR_MAP,
)
apt.pl.add_lines(ax=axm[0], intercepts=0, linetype="vline")
apt.pl.add_lines(ax=axm[0], intercepts=0, linetype="hline")

Samples separate by severity of the disease in PC1.

## Differential expression analysis

Use a ttest to identify differentially expressed proteins.

We subset to the respective conditions and restrict the analysis to proteins that with sufficient evidence (>70% completeness) in the two conditions.

In [ ]:
adata_agg_ten = adata_agg[adata_agg.obs["condition"].isin(["Healthy", "TEN"])].copy()
adata_agg_ten = apt.pp.filter_data_completeness(
    adata_agg_ten,
    max_missing=0.3,
    group_column="condition",
    groups=["Healthy", "TEN"],
    keep_strategy="all",
    action="drop",
)

In [ ]:
diff_exp = apt.tl.diff_exp_ttest(
    adata_agg_ten, between_column="condition", comparison=("TEN", "Healthy")
)
diff_exp = diff_exp.merge(adata.var, left_index=True, right_index=True)
diff_exp = diff_exp.set_index("genes")

In [ ]:
dc.pl.volcano(
    diff_exp,
    x="log2fc",
    y="fdr",
    top=10,
    thr_stat=1,
    color_pos="#CBA7CE",
    color_neg="#999FA3",
    color_null="#cccccc",
)
plt.xlabel("$\log_{2}{FC}$\nHealthy $\longleftrightarrow$ TEN")

Most upregulated in TEN vs. healthy individuals are WARS1 (IFN-gamma response), STAT1, TAPBP and other the histocompatibility complex class I proteins. 

## Integration with spatial information

For the integration of proteomics data with the spatial component it is necessary to map the measurement back to a specific spatial location. This is not possible with the given data, as it represents minibulk measurements of specifically dissected cells across a whole tissue section. 

If there is a direct mapping between MS-measurements and spatial information possible, users can add the obtained anndata object to the spatialdata object of the [image analysis](../image-analysis/harpy.ipynb). 

Further tutorials on this are available in:

- [alphapepttools](https://mannlabs.github.io/alphapepttools/notebooks/studies/study_04_scDVP.html)
- [openDVP](https://coscialab.github.io/openDVP/Tutorials/T3_ProteomicsIntegration.html)
- [dvp-io](https://dvp-io.readthedocs.io/en/latest/tutorials/004_scdvp.html)

## Session Info 

In [ ]:
from session_info2 import session_info

session_info(dependencies=True)